In [17]:
class Bank():
  _country = "KOREA"

  def __init__(self, budget, name):
    self.__budget = budget
    self.__name = name
  
  @property
  def budget(self):
    return self.__budget

  @budget.setter
  def budget(self, budget):
    if budget < 0:
      print("음수 탐지")
      return False
    else:
      self.__budget = budget

  def get_name(self):
    return self.__name
  
class MyBank(Bank):
  
  def __init__(self, budget, name):
    super().__init__(budget, name)
  
  def info(self):
    print(f"은행 이름: {self.get_name()}")
    print(f"나라 이름: {Bank._country}")
    print(f"예산: {self.budget}")

mb = MyBank(10000, "hur")


print(mb._country)
mb._country="asdf"

mb.budget = -2000

mb.info()
# print(mb._Bank__budget)


KOREA
음수 탐지
은행 이름: hur
나라 이름: KOREA
예산: 10000


In [ ]:
import datetime

class BankAccount:
  COUNTRY = "KOREA"
  BANK_NAME = "GPT Bank"

  def __init__(self, owner, balance):
    # 외부에서 쉽게 접근할 수 없게 private로 mangling
    self.__owner = owner
    self.__balance = balance

  @property
  def owner(self):
    return self.__owner
  
  @owner.setter
  def owner(self, owner: str):
    if not owner or owner == "":
      print("이름 오류")
      return
    
    self.__owner = owner
  
  @property
  def balance(self):
    # return 0
    return self.__balance
  
  @balance.setter
  def balance(self, balance: int):
    if balance < 0:
      print("잔액 오류")
      return
    self.__balance = balance
  
  def _validate_amount(self, amount):
    if amount >= 0:
      return True
    print("금액 오류")
    return False

  def deposit(self, amount):
    if amount < 0:
      print("입금 실패")
      return False
    self.__balance += amount
    print(f"입금 완료: {amount}원")
    return True

  def withdraw(self, amount):
    if amount < 0:
      print("출금 실패")
      return False
    elif self.__balance < amount:
      print("잔액 부족")
      return False
    self.__balance -= amount
    print(f"출금 완료: {amount}원")
    print(f"남은 금액: {self.__balance}")
    return True

class AuditMixin(BankAccount):

  def __init__(self, owner, balance, logs):
    super().__init__(owner, balance)
    self._logs = logs
  
  def deposit(self, amount):
    self._logs.append(f"[{datetime.datetime.now()}] deposit 요청: {amount}원")
    result = super().deposit(amount)

    self._logs.append(f"[{datetime.datetime.now()}] deposit 종료: {amount}원")
    return result

  def withdraw(self, amount):
    self._logs.append(f"[{datetime.datetime.now()}] withdraw 요청: {amount}원")
    result = super().deposit(amount)

    self._logs.append(f"[{datetime.datetime.now()}] withdraw 종료: {amount}원")
    return result

class FeeMixin(BankAccount):
  WITHDRAW_FEE = 500

  def withdraw(self, amount):

    if input("수수료 500원이 적용됩니다. 적용 하시겠습니까? (y/n)").lower() == "y":
      return False
    print(f"수수료 500원이 적용되었습니다. 총 출금액: {amount+500}")

    result = super().withdraw(amount+500)

    return result

class LimitMixin(BankAccount):
  # 출금 한도
  MAX_WITHDRAW = 5000
  
  def __init__(self, owner, balance):
    super().__init__(owner, balance)

  def withdraw(self, amount):
    if amount > self.MAX_WITHDRAW:
      print(f"1회 출금 한도를 벗어났습니다. 출금 한도: {self.MAX_WITHDRAW}")
      return False
    
    return super().withdraw(amount)

# 항상 로깅을 먼저 하며
# 그 다음으로 한도초과 확인 후
# 수수료 때기
class SmartAccount(AuditMixin, LimitMixin, FeeMixin, BankAccount):
  def __init__(self, owner, balance):
    self._logs = []
    super().__init__(owner, balance, self._logs)
  
  def info(self):
    print(f"소유자: {self.owner}")
    print(f"잔액: {self.balance}")
    print(f"은행: {BankAccount.BANK_NAME}")
    print(f"국가: {BankAccount.COUNTRY}")
    


In [28]:
acc = SmartAccount("hur", 10000)

print(SmartAccount.mro())

acc.info()

acc.deposit(3000)
print(acc.balance)  # 13000

# acc.withdraw(2000)
# print(acc.balance)  # 수수료 포함해서 10500이어야 함

# acc.withdraw(6000)  # 1회 출금 한도 초과

# acc.balance = -1000
# print(acc.balance)  # 기존 잔액 유지

# acc.owner = ""
# print(acc.owner)  # 기존 이름 유지

# acc.COUNTRY = "USA"
# acc.info()  # 국가는 여전히 KOREA 출력

# print(acc.__dict__)
# print(acc._BankAccount__balance)  # name mangling 확인용
# print(acc._logs)

[<class '__main__.SmartAccount'>, <class '__main__.AuditMixin'>, <class '__main__.LimitMixin'>, <class '__main__.FeeMixin'>, <class '__main__.BankAccount'>, <class 'object'>]
소유자: hur
잔액: 0
은행: GPT Bank
국가: KOREA
입금 완료: 3000원
0
